# Quality Control Tutorial

Learn to use the `PlantCV` quality control submodule (`pcv.qc`) to evaluate your images before and after processing.

This tutorial covers:
- Checking for over- or underexposed images with `pcv.qc.exposure`
- Detecting and evaluating color cards for color quality assessment
- Comparing observed and expected color chip values with `pcv.qc.quick_color_check`
- Visualizing color chip fidelity with `pcv.qc.color_chip_comparison`
- Measuring and plotting perceptual color difference (delta E) with `pcv.transform.deltaE` and `pcv.qc.plot_deltaE`
- Applying and evaluating linear color correction with `pcv.transform.affine_color_correction`
- Assessing color quality across a whole dataset of images

In [ ]:
# Install PlantCV and required dependencies
%pip install "altair>=5" plantcv

# Give access and mount your Google Drive (need a Google Account)
# Change path to directory you wish output files to be saved to.
from google.colab import drive
drive.mount('/content/gdrive')

# Enable widget feature with matplotlib
from google.colab import output
output.enable_custom_widget_manager()

#View working directory, may need to change path
%pwd

### DEV NOTE:
Copy the next code cell if the tutorial you are working on meets any of the following:
* use of a file type that is not JPG/PNG/TIFF
* requires a batch of images

Users will need to clone the repository to their Google Drive and change the working directory to the cloned repository. This ensures none of the file paths need to be altered in the tutorial notebook.

In [ ]:
# Change your working directory to the mounted drive 
%cd gdrive/MyDrive/ 
# Print the contents of your drive to confirm it worked 
!ls 
# Clone the workshop's repository to your Google Drive 
!git clone https://github.com/danforthcenter/plantcv-tutorial-qc.git
# Change directory to the cloned repository
%cd plantcv-tutorial-qc

# Section 1: Importing Image and Libraries

In [ ]:
# Set the notebook display method
# If widget is not working, then change to inline
%matplotlib widget

# Import plantcv and the parallel workflow helper
from plantcv import plantcv as pcv
from plantcv.parallel import WorkflowInputs

# Print the version of PlantCV being used by the Jupyter kernel
pcv.__version__

## Input/Output variables

WorkflowInputs mimics the workflow command-line argument parser that is used for workflow parallelization. Using it while developing a workflow in Jupyter makes it easier to convert the workflow to a script later.

In [ ]:
# Input/output options
args = WorkflowInputs(
    images=["img/image1.png"],  # list of image file paths to process
    names="image1",             # variable name(s) for each image (accessible as args.image1, etc.)
    result="test.json",         # output file path for storing analysis results
    outdir=".",                 # directory to write debug/output images
    debug="plot"                # debug mode: "plot" displays inline, "print" saves to outdir, None disables
    )

In [ ]:
# Set debug to the global parameter 
pcv.params.debug = args.debug
# dpi controls the resolution of saved/displayed debug images
pcv.params.dpi = 100
# text_size and text_thickness control label readability in debug output images
# increase for large images; decrease for small images
pcv.params.text_size = 1
pcv.params.text_thickness = 1

## Read the input image

### Reading images into your environment using *pcv.readimage()*

`pcv.readimage` loads an image from disk and returns the image array along with its directory path and filename. The `mode` parameter controls how the image data is interpreted:

**Inputs:**
- `filename` = path to the image file to read
- `mode` = how the image is read into memory (`'native'` default)

**Outputs** (returned as a tuple):
- `img` = image data as a numpy array
- `path` = directory path of the image
- `filename` = filename of the image

In [ ]:
# Read the image; returns the array (BGR), its directory path, and its filename
img, path, filename = pcv.readimage(filename=args.image1)

# Section 2: Overall Image Quality

The first step in quality control is often to evaluate the overall quality of the image. Spefically here we want to make sure that our image is capturing reasonable ranges of color by looking at exposure.
There are many other aspects to what makes a good image and for those we would refer to [guidelines from the DDPSC phenotyping core facility](https://www.jove.com/t/67619/imaging-analysis-for-quantifying-maize-zea-mays-abiotic-stress)

## Check image exposure using *pcv.qc.exposure*

`pcv.qc.exposure` plots the pixel intensity distribution for each RGB channel as a histogram and checks whether the image is over- or underexposed.

**How it works:**
For each channel (red, green, blue), the function counts the fraction of pixels at the minimum (0) or maximum (255) intensity. Pixels that "clip" at either extreme have lost actual color information — very dark shadows and very bright highlights are indistinguishable. If any channel has more than `warning_threshold` of its pixels clipped, a warning is raised and the per-channel proportions are saved to `pcv.outputs.metadata`.

**Inputs:**
- `rgb_img` = an RGB image as a numpy array
- `warning_threshold` = fraction of pixels allowed to be at 0 or 255 before a warning fires (default `0.05`, i.e., 5%)

**Output:**
- An Altair chart with three side-by-side histograms (one per channel), showing the proportion of pixels at each intensity value 0–255

In [ ]:
# Check image exposure
# rgb_img: the image to evaluate (pcv.readimage returns BGR, but exposure() handles the conversion internally)
# warning_threshold=0.05: warn if >5% of pixels in any channel are fully black (0) or fully white (255)
# Lower the threshold for stricter exposure checking; raise it to allow more latitude
hist = pcv.qc.exposure(rgb_img=img, warning_threshold=0.05)

# Section 3: Color Card Quality

Most `pcv.qc` functions are designed to assess color accuracy in an image. This typically requires a **color reference card** — a physical target with chips of known colors — so that you have ground-truth expectations for what colors in the image should look like.

**Why use a color card?**
Camera sensors, lighting conditions, and lens characteristics all introduce color bias. A color card lets you measure how much your camera's output deviates from the true color and, if needed, mathematically correct it.

To use these tools, we first detect the color card in the image with `pcv.transform.detect_color_card`. This returns a matrix of observed color chip values (mean R, G, B per chip) that can be compared against expected (standard) values.

**Why evaluate color card quality?**
A color card, like any physical hardware, has a finite lifespan. Exposure to light, oxygen, water, and other wear-and-tear that can accumulate over the course of many experiments can degrade the fidelity of the color chips in any color card. If we are worried about accurately portraying color or comparing perceptual differences then having an accurate target to correct to is important. Unfortunately, due to differences in imaging environments and cameras it can be hard to get an objective measurement of color in a color card. We recommend using your best judgement over any hard and fast cutoff for determining color card health, but these tools can help with that process.

**First step, detecting a color card**

`pcv.transform.detect_color_card` returns a matrix of color chip locations and mean RGB values (scaled 0–1). By default it also calculates perceptual color difference (delta E CIEDE2000) for each chip and stores the values in `pcv.outputs.metadata` under the key `"deltaE_uncalibrated"`.

This matrix can be used in several downstream QC functions. The first is `pcv.qc.quick_color_check`, which plots observed channel values against expected values from a reference matrix. This scatter plot lets you quickly assess whether the camera records each channel linearly.


In [ ]:
# Detect the color card in the image and extract per-chip mean color values
# color_chip_size="passport": expects a passport-sized Macbeth ColorChecker card
#   other options: None (auto-detect chip size), or a (width, height) tuple in pixels
# Returns an N x 4 matrix: [chip_id, R_mean, G_mean, B_mean] for each chip, values scaled 0–1
# Also stores deltaE_uncalibrated in pcv.outputs.metadata by default (set delta_E=False to skip)
cc_mat = pcv.transform.detect_color_card(img, color_chip_size="passport")

## Comparing chip colors visually with *pcv.qc.color_chip_comparison*

`pcv.qc.color_chip_comparison` provides a visual check by displaying the **actual rendered colors** of each chip side-by-side — a larger left swatch showing the observed color and a smaller right swatch showing the expected color — for any number of input matrices.

**How chips are arranged:**
Chips are sorted along the Y-axis by "greenness" — the ratio G / (R + G + B). This metric is used as a heuristic proxy for color card physical condition: while perceptual color differences (delta E) can shift with lighting or camera settings, the relative proportion of green light each chip reflects is more strongly tied to the physical state of the card itself. Faded chips — which in the Macbeth card tend to affect the red chips first — will change their greenness rank relative to the standard.

The lower panel shows observed greenness rank vs. expected greenness rank. Points on the diagonal line indicate a chip's relative greenness is consistent with the standard; deviations flag chips that may be physically degraded.

**Inputs:**
- `std_matrix` = expected reference matrix (output of `pcv.transform.std_color_matrix`), N x 4 scaled 0–1
- `*args` = one or more observed color matrices (output of `pcv.transform.detect_color_card`); pass multiple to compare across sessions or correction methods

In [ ]:
# Build the standard (expected) reference matrix for a 24-chip Macbeth ColorChecker
# pos=3 matches the orientation returned by detect_color_card
# Returns an N x 4 matrix: [chip_id, R, G, B], scaled 0–1
std_mat = pcv.transform.std_color_matrix(pos=3)

# Compare observed chip colors to expected chip colors
# std_mat: N x 4 expected reference matrix [chip_id, R, G, B], scaled 0–1 (first positional arg)
# cc_mat: one or more N x 4 observed matrices passed as *args; add more matrices to compare sessions
# Returns a vertically concatenated Altair chart: color swatch panel (top) + greenness rank scatter (bottom)
ccc_plot = pcv.qc.color_chip_comparison(std_mat, cc_mat)
ccc_plot

# Section 4: Image Color Quality

The rest of the `pcv.qc` tools are more focused on how you should approach color correction and how perceptually different the colors in your (raw or corrected) image are from the "true colors" in a color card.

## Quick Color Checks

Earlier we made a target matrix using `pcv.transform.std_color_matrix` as an example, but if no target is provided `pcv.qc.quick_color_check` defaults to `std_color_matrix` for a 24-chip Macbeth card or `astro_color_matrix` for a 15-chip AstroBotany card, chosen automatically based on the shape of the source matrix.

**Interpreting the plot:**
Ideally each channel produces a straight line with slope 1 and intercept 0, meaning observed = expected. Curved (non-linear) lines suggest the camera has a non-linear response in that channel, which indicates that a linear (affine) color correction may not be sufficient — you might consider a polynomial or per-channel lookup-table approach instead.

In [ ]:
# Plot observed (source) vs. expected (target) channel values for each chip
# source_matrix: N x 4 observed matrix from detect_color_card [chip_id, R, G, B]
# target_matrix: N x 4 expected reference matrix; if None, defaults to std or astro matrix automatically
# num_chips: number of chips to include; defaults to the row count of target_matrix
qcc_plot = pcv.qc.quick_color_check(source_matrix=cc_mat)

## Delta E values in outputs metadata

By default, `pcv.transform.detect_color_card` (and the auto correction wrappers) automatically compute delta E (CIEDE2000) for each color chip and store the results in `pcv.outputs.metadata` as a list under the key `"deltaE_uncalibrated"`. Setting `delta_E=False` when calling `detect_color_card` will skip this measurement.

**What is delta E?**
Delta E (ΔE) is a perceptual color difference metric. A value of 0 means the observed color is identical to the expected color; higher values indicate greater perceptual divergence. CIEDE2000 is the most perceptually uniform version of this metric and is the standard used throughout PlantCV.

In [ ]:
pcv.outputs.metadata["deltaE_uncalibrated"]

This data can be used in downstream analysis to check image quality or compare color correction methods.

`pcv.transform.deltaE` can also be called directly to compute delta E as a numpy matrix. The `obs` argument sets the suffix appended to the `"deltaE_"` key in `pcv.outputs.metadata`, so values from different processing steps can be stored and compared side-by-side (e.g. `"deltaE_uncalibrated"` vs `"deltaE_affine_color_correction"`).

`pcv.qc.plot_deltaE` takes that matrix and renders an interactive chart where each chip is colored by a perceptual quality bin:

| Delta E range | Interpretation |
|---|---|
| < 1 | Imperceptible difference |
| 1–2 | Perceptible only under close inspection |
| 2–10 | Perceptible difference |
| 10–49 | Colors are clearly different |
| > 49 | Colors are opposite hues |

Whether a particular chip's delta E matters for your experiment depends on your analysis plan — chips in spectral regions irrelevant to your trait of interest may be acceptable even at higher delta E values.

We temporarily set `pcv.params.debug = None` before calling `pcv.transform.deltaE` to suppress the verbose color card detection overlay, then re-enable plotting before `pcv.qc.plot_deltaE`.

In [ ]:
pcv.params.debug = None  # suppress color card detection debug image
# Calculate delta E (CIEDE2000) per chip for the uncorrected image
# arg 1: image to evaluate (numpy array)
# arg 2: "passport" specifies the Macbeth passport card size/type for detection
# obs="uncalibrated": suffix for the metadata key → stored as "deltaE_uncalibrated"
de_mat = pcv.transform.deltaE(img, "passport", obs="uncalibrated")
pcv.params.debug = "plot"  # re-enable plotting for the QC chart
# Plot per-chip delta E as a bar chart colored by perceptual quality bin
# source: numpy.ndarray of delta E values → single-image bar chart
p = pcv.qc.plot_deltaE(de_mat)

### Color Correction

Now that we have assessed image quality, we can apply color correction to bring the observed colors closer to their expected values.

**Choosing a correction method:**
The `pcv.qc.quick_color_check` plot showed a roughly linear relationship between observed and expected channel values across all three channels, so we use `pcv.transform.affine_color_correction` — a linear (matrix multiplication) correction that adjusts for systematic camera bias.

If `quick_color_check` had revealed curved (non-linear) relationships in any channel, you would instead consider a per-channel polynomial correction or another non-linear approach.

In [ ]:
# Apply affine (linear) color correction to the image
# img: the original uncorrected image (numpy array)
# cc_mat: observed color chip matrix from detect_color_card [chip_id, R, G, B], scaled 0–1
# std_mat: expected reference matrix from std_color_matrix [chip_id, R, G, B], scaled 0–1
# Returns the color-corrected image as a numpy array
# Also automatically computes delta E on the corrected image and stores it as
# "deltaE_affine_color_correction" in pcv.outputs.metadata
img_cc = pcv.transform.affine_color_correction(img, cc_mat, std_mat)

`pcv.transform.affine_color_correction` automatically measures delta E on the corrected image and stores the values in `pcv.outputs.metadata` under the key `"deltaE_affine_color_correction"`. This makes it straightforward to compare pre- and post-correction color fidelity in your downstream analysis — both keys will be present in the metadata output.

In [ ]:
pcv.outputs.metadata["deltaE_affine_color_correction"]

We can call `pcv.qc.plot_deltaE` again on the corrected image to evaluate how much the correction improved color fidelity. We expect most bars to drop into lower (greener) delta E bins compared to the uncalibrated plot above.

In this example the values are substantially reduced — most chips are now below 10, with several approaching 1–2 (nearly imperceptible difference from the expected color). The improvement confirms that affine correction was a good choice for this image and camera setup.

In [ ]:
pcv.params.debug = None  # suppress debug image for deltaE detection
# Compute delta E for the corrected image
# No obs argument: metadata key defaults to "deltaE_calibrated"
de_mat = pcv.transform.deltaE(img_cc, "passport")
pcv.params.debug = "plot"
# Plot per-chip delta E values for the corrected image to compare with the pre-correction chart
p = pcv.qc.plot_deltaE(de_mat)

In [ ]:
# we'll clear outputs to reset for next steps
pcv.outputs.clear()

#### Using the auto-correction wrapper

`pcv.transform.auto_correct_color` is a convenience wrapper that combines color card detection, color matrix extraction, and affine color correction in a single call. It replicates the manual steps above (`detect_color_card` → `std_color_matrix` → `affine_color_correction`) but requires less boilerplate.

The wrapper also automatically collects delta E for both the uncalibrated and corrected image, so `pcv.outputs.metadata` will contain two delta E entries after calling it — one for each stage.

In [ ]:
# Auto color correction: detects the card, builds the reference matrix, and applies affine correction
# arg 1: the original uncorrected image
# "passport": color card size/type — same options as pcv.transform.detect_color_card
# Returns the color-corrected image; populates pcv.outputs.metadata with two delta E entries
img_cc2 = pcv.transform.auto_correct_color(img, "passport")

In [ ]:
print(pcv.outputs.metadata.keys()) # contains 2 delta E versions
pcv.outputs.clear()

### Dataset-level Color Quality Assessment

So far we have focused on quality control for a single image — which is valuable while developing a workflow, tuning detection parameters, and setting thresholds. Once you have a larger imaging dataset, you may want to survey color quality across many images at once.

`pcv.qc.plot_deltaE` accepts a **directory path string** or a **list of file paths** in addition to a single numpy array. When given a directory, it automatically discovers images (including subdirectories), detects color cards, computes per-chip delta E for each image, and returns a **boxplot** with per-image points overlaid.

This is particularly useful for:
- Checking lighting consistency across a time series or multi-camera setup
- Identifying outlier images with unusually high delta E (poor lighting, camera drift, or dirty lens)
- Deciding whether per-image or per-batch color correction is needed

**Inputs (directory/list mode):**
- `source` = str path to a directory of images, or a list of file paths
- `n` = maximum number of images to load from a directory (default `20`)
- `ext` = file extension to search for (default `"png"`)
- `**kwargs` = forwarded to `pcv.transform.deltaE` (e.g. `color_chip_size`, `roi`, `adaptive_method`)

In [ ]:
# Compute and plot delta E across all images in the "img" directory
# source="img": directory to search for images (recurses into subdirectories)
# n=20: maximum images to load (default); reduce for large datasets or increase for full surveys
# ext="png": only process files with this extension (default)
# When source is a str/list → returns a boxplot with individual image points per chip
p = pcv.qc.plot_deltaE(
    source="img",
    # n=20,      # max images to process
    # ext="png", # file extension filter
)
p

In [ ]:
print(pcv.outputs.metadata.keys())

# Conclusion

The `pcv.qc` submodule provides a suite of tools for evaluating image quality before and after processing:

- `pcv.qc.exposure` — checks for over- or underexposed channels that would compromise color analysis
- `pcv.qc.quick_color_check` — visualizes the linearity of the camera's color response against a reference, helping you choose a correction method
- `pcv.qc.color_chip_comparison` — renders observed vs. expected chip colors and uses greenness rank to detect physical card degradation
- `pcv.qc.plot_deltaE` — quantifies perceptual color error per chip, either for a single image (bar chart) or an entire dataset (boxplot), colored by standard interpretability thresholds

Together these tools help ensure your images are suitable for analysis and that any color correction is performing as expected.

If you have ideas for improvements to the `qc` submodule, feature requests, or contributions to any part of PlantCV, we encourage you to open an [issue](https://github.com/danforthcenter/plantcv/issues) on GitHub or contribute directly with a [pull request](https://docs.plantcv.org/en/stable/CONTRIBUTING/).